In [2]:
import anndata as ad
import squidpy as sq
import cellcharter as cc
import pandas as pd
import scanpy as sc
import scvi
import numpy as np
import matplotlib.pyplot as plt
from lightning.pytorch import seed_everything
from pathlib import Path
from skimage.io import imread

import torch

/home/jon/anaconda3/envs/spatial_analysis_env/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/jon/anaconda3/envs/spatial_analysis_env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/jon/anaconda3/envs/spatial_analysis_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# COMPUTER
segmentation_path = Path("/mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/data/cell_segmentation/")
adata_file     = segmentation_path / "concatenated" / "combined_adata.h5ad"
geneList = segmentation_path / "Gene_lists"
fig_out = Path("/Users/janzules/Roselab/Spatial/CAR_T/figures/clustering_results/")
ST_sample = sc.read_h5ad(adata_file)

In [11]:
ST_sample.obsm["spatial"] = ST_sample.obs[["cx", "cy"]].to_numpy()

In [28]:
ST_sample.obs['mouse'].unique()

['RTCyT72_2_1', 'RTCyPSCA_1_4', 'CyPSCA_1_1', 'NoTx_2_2', 'CyPSCA_1_2', 'RTCyT72_2_4', 'RTCyPSCA_2_4', 'CyT72_1_4']
Categories (8, object): ['CyPSCA_1_1', 'CyPSCA_1_2', 'CyT72_1_4', 'NoTx_2_2', 'RTCyPSCA_1_4', 'RTCyPSCA_2_4', 'RTCyT72_2_1', 'RTCyT72_2_4']

In [29]:
ST_subset = ST_sample[ST_sample.obs['mouse'].isin(['NoTx_2_2'])].copy()

In [30]:
import squidpy as sq
import scanpy as sc

# make sure coords exist
assert "spatial" in ST_subset.obsm

# build spatial graph
# For Visium/Visium HD spots use coord_type="grid"; otherwise "generic"
sq.gr.spatial_neighbors(ST_subset, coord_type="generic")  # or coord_type="generic"

# Moran’s I for one gene
res = sq.gr.spatial_autocorr(
    ST_subset,
    mode="moran",           # or "geary"
    genes=["Glp1r"],
    n_perms=1000
)
print(ST_subset.uns["moranI"].loc["Glp1r"])


100%|████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:04<00:00, 212.67/s]

I                    2.220756e-02
pval_norm            2.220446e-16
var_norm             7.449237e-06
pval_z_sim           0.000000e+00
pval_sim             9.990010e-04
var_sim              4.596781e-06
pval_norm_fdr_bh     2.220446e-16
pval_z_sim_fdr_bh    0.000000e+00
pval_sim_fdr_bh      9.990010e-04
Name: Glp1r, dtype: float64


I                    0.037565
pval_norm            0.000000
var_norm             0.000017
pval_z_sim           0.000000
pval_sim             0.000999
var_sim              0.000008
pval_norm_fdr_bh     0.000000
pval_z_sim_fdr_bh    0.000000
pval_sim_fdr_bh      0.000999
Name: Glp1r, dtype: float64
